In [ ]:
import io 
import os 
import sys
import h5py 
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.cbook as cbook
import matplotlib.cm as cm
import scipy
import csv
import pandas as pd
from pathlib import Path
from datetime import date, datetime, timedelta
import matplotlib.backends.backend_pdf
from matplotlib.ticker import (MultipleLocator, AutoMinorLocator)

import matplotlib.ticker as ticker
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns

svdir = '/Users/christiandewey/Library/CloudStorage/GoogleDrive-christian.w.dewey@gmail.com/My Drive/manuscripts/2023_Dewey-Fendorf-etal_meanders/plots/'


xpltl = -0.1
ypltl = 0.98
plt.rc('font', size=8) 
plt.rcParams['axes.labelsize'] = 10
plt.rcParams['mathtext.default'] = 'rm'
bwidth = 0.75

In [ ]:
def makePanel(component,data,years,ax):    
    ax = sns.boxplot(x = "trans_pos_m", y = component,
        data = data,ax=ax, width = bwidth,hue ='year',showfliers = False,linewidth=0.8,palette= ['dimgrey','gainsboro'],notch=False, showcaps=False,medianprops={"alpha": 0.0})  #'dimgrey'
    ax.set_xticklabels(ax.get_xticklabels(),rotation=90, horizontalalignment= 'center')
    leg = ax.get_legend().remove()
    if 2017 in years:
        palette_y = ['viridis','rocket']
    elif 2019 in years:
        palette_y = ['rocket','viridis']
    for year,offset,pal in zip(years,[-0.3,0.3],palette_y):
        df = data[data['year']==year].copy()
        df['CatPos'] = df['CatPos'] + offset
        ax = sns.scatterplot(data=df, x='CatPos', y=component, hue='Date',
                        palette=pal, size=4,ax=ax, edgecolor = None, legend=False)
        ax.set_xlabel('Location')
        ax.yaxis.set_minor_locator(AutoMinorLocator())
    sns.despine()
    
    
def makeColorBars(years, fig):

    print(years)
    if 2017 in years:
        palette_y = ['viridis','rocket']
    elif 2019 in years:
        palette_y = ['rocket','viridis']
    cbar_ax = fig.add_axes([1.00, 0.55, 0.02, 0.3])
    y1 = years[0]
    mind =  date(int(y1), 5, 25)  #min(sub['Date'])
    mindf = mind.timetuple().tm_yday
    maxd = date(int(y1), 10, 25)
    maxdf = maxd.timetuple().tm_yday
    nValues = np.arange(mindf,maxdf)
    norm = matplotlib.colors.Normalize(vmin=mindf, vmax=maxdf) 
    scalarmappaple = cm.ScalarMappable(norm = norm, cmap=palette_y[0])
    scalarmappaple.set_array(nValues)
    cb = fig.colorbar(scalarmappaple, orientation='vertical',cax=cbar_ax,shrink=0.5)
    cb_labels = [i.get_text() for i in cb.ax.get_yticklabels()] 
    startdate = date(int(y1), 1, 1)
    labels = [(timedelta(days = float(day_num) -1.0) + startdate) for day_num in cb_labels]
    cb.ax.set_yticklabels( [d.strftime("%-d %b %y") for d in labels],size = 8 )

    cbar2_ax = fig.add_axes([1.0, 0.15, 0.02, 0.3])
    y2 = years[1]
    mind =  date(int(y2), 5, 25)  #min(sub['Date'])
    mindf = mind.timetuple().tm_yday
    maxd = date(int(y2), 10, 25)
    maxdf = maxd.timetuple().tm_yday
    nValues = np.arange(mindf,maxdf)
    norm = matplotlib.colors.Normalize(vmin=mindf, vmax=maxdf) 
    scalarmappaple = cm.ScalarMappable(norm = norm, cmap=palette_y[1])
    scalarmappaple.set_array(nValues)
    cb2 = fig.colorbar(scalarmappaple, orientation='vertical',cax=cbar2_ax,shrink=0.5)
    cb2_labels = [i.get_text() for i in cb2.ax.get_yticklabels()] 
    startdate = date(int(y2), 1, 1)
    labels = [(timedelta(days = float(day_num) -1.0) + startdate) for day_num in cb2_labels]
    cb2.ax.set_yticklabels( [d.strftime("%-d %b %y") for d in labels],size = 8 )  


In [ ]:
# process data to produce 2017 + 2018 MCP grids
# pH, SO4, TIC, Ca, DOC, Fe, Sr, Cl, Si + river concs + river elevations
fdir = '/Users/christiandewey/Library/CloudStorage/GoogleDrive-christian.w.dewey@gmail.com/My Drive/manuscripts/2023_Dewey-Fendorf-etal_meanders/data'

t = 'MCP'
positions = {'MCP1-1': 2, 'MCP1-2':17, 'MCP1-3': 32, 'MCP1-4': 48, 'MCP1-5': 63 }
positions = {'MCP1-1': "ER-MCP1", 'MCP1-2':"ER-MCP2", 'MCP1-3': "ER-MCP3", 'MCP1-4': "ER-MCP4", 'MCP1-5': "ER-MCP5" }
positions = {'MCP1-1': "MCP1", 'MCP1-2':"MCP2", 'MCP1-3': "MCP3", 'MCP1-4': "MCP4", 'MCP1-5': "MCP5" }

'''datemin = np.datetime64("2017-02-25")
datemax = np.datetime64("2017-10-15")'''


## get 2017 MCP data
fname = '/porewater/mc_2017_porewater_ess-dive.csv'
data1 = pd.read_csv(fdir + fname,parse_dates=['Date'])
data1['trans_pos_m'] = None

for r in range(len(data1)):
    w = data1['Well'].iloc[r]
    if t in w:
        data1.loc[r,('trans_pos_m')]= positions[w]
    
sub17 = data1[data1['Well'].str.contains(t)==True].copy()
sub17['Ca_M'] = sub17['Ca_M']*1e3
sub17['Fe_M'] = sub17['Fe_M'].loc[:]*1e6
sub17['P_M'] = sub17['P_M']*1e6
sub17['SO4_mM'] = sub17['SO4_mM']*1e3
sub17['NO3_mM'] = sub17['NO3_mM']*1e3
sub17['NH4_mM'] = sub17['NH4_mM']*1e3
sub17['DOC_mM'] = sub17['DOC_mM']*1e3


## get 2018 MCP data
fname = '/porewater/mc_2018_porewater_ess-dive.csv'
data2 = pd.read_csv(fdir + fname,parse_dates=['Date'])
data2['trans_pos_m'] = None

for r in range(len(data2)):
    w = data2['Well'].iloc[r]
    if t in w:
        data2.loc[r,('trans_pos_m')]= positions[w]
    
sub18 = data2[data2['Well'].str.contains(t)==True].copy()
sub18['Ca_M'] = sub18['Ca_M']*1e3
sub18['Fe_M'] = sub18['Fe_M'].loc[:]*1e6
sub18['P_M'] = sub18['P_M']*1e6
sub18['SO4_mM'] = sub18['SO4_mM']*1e3
sub18['NO3_mM'] = sub18['NO3_mM']*1e3
sub18['NH4_mM'] = sub18['NH4_mM']*1e3
sub18['DOC_mM'] = sub18['DOC_mM']*1e3

i = 0
for c, d in zip(sub18.columns,sub17.columns):
    if c != d :
        print(c,d, i)
        break
    else:
        i = i+1

mcp17_18 = pd.concat([sub17,sub18],ignore_index=True)
mcp17_18['year'] = mcp17_18['Date'].dt.year.copy()
catpos = {'MCP1-1': 0, 'MCP1-2':1, 'MCP1-3': 2, 'MCP1-4': 3, 'MCP1-5': 4 }

def func(x):
    v = catpos[x]
    return v

mcp17_18['CatPos'] = mcp17_18['Well'].map(lambda x: func(x))

In [ ]:
# create MC plots
plt.rcParams['ytick.major.pad']='1.5'
plt.rcParams['xtick.major.pad']='0.5'
xpltl = -.25
ypltl = 0.98

years = [2017,2018]

fig, ((ax1,ax3),(ax2,ax7),(ax5,ax8)) = plt.subplots(3,2, figsize = (6, 8))

makePanel('pH',mcp17_18,years, ax1)
ax1.set_ylabel('pH')
ax1.yaxis.set_major_locator(MultipleLocator(1))
ax1.set_ylim(6.6,8.3)

makePanel('Fe_M',mcp17_18,years, ax2)
ax2.set_ylabel('Fe ('+r'$\mu$M)')
ax2.set_ylim(bottom = 0)

makePanel('NH4_mM',mcp17_18,years, ax3)
ax3.set_ylabel('NH$_4^{+}$ ('+r'$\mu$M)')
ax3.yaxis.set_minor_locator(AutoMinorLocator())
ax3.set_ylim(bottom = 0)

makePanel('SO4_mM',mcp17_18,years, ax7)
ax7.set_ylabel('SO$_4^{2-}$ ('+r'$\mu$M)')
ax7.yaxis.set_minor_locator(AutoMinorLocator())
ax7.set_ylim(bottom = -5)

makePanel('DIC_mM',mcp17_18,years, ax5)
ax5.set_ylabel('DIC (mM)')
ax5.yaxis.set_major_locator(MultipleLocator(1))
ax5.yaxis.set_minor_locator(AutoMinorLocator())
ax5.set_ylim(bottom = 0)

makePanel('DOC_mM',mcp17_18,years, ax8)
ax8.set_ylabel('DOC ('+r'$\mu$M)')
ax8.set_ylim(bottom = 0)

ax = ax1 #pH 
ax.text(xpltl, ypltl,'a',
        horizontalalignment='center',
        verticalalignment='center',
        transform = ax.transAxes, fontweight='bold', fontsize = 12)

ax =ax3 # ammonium
ax.text(xpltl, ypltl,'b',
        horizontalalignment='center',
        verticalalignment='center',
        transform = ax.transAxes, fontweight='bold', fontsize = 12)

ax =ax5 # DIC
ax.text(xpltl, ypltl,'e',
        horizontalalignment='center',
        verticalalignment='center',
        transform = ax.transAxes, fontweight='bold', fontsize = 12)

ax = ax2 #Fe
ax.text(xpltl, ypltl,'c',
        horizontalalignment='center',
        verticalalignment='center',
        transform = ax.transAxes, fontweight='bold', fontsize = 12)
ax = ax8 #DOC
ax.text(xpltl, ypltl,'f',
        horizontalalignment='center',
        verticalalignment='center',
        transform = ax.transAxes, fontweight='bold', fontsize = 12)

ax = ax7 #sulfate
ax.text(xpltl, ypltl,'d',
        horizontalalignment='center',
        verticalalignment='center',
        transform = ax.transAxes, fontweight='bold', fontsize = 12)



makeColorBars([2017,2018],fig)

fig.tight_layout()

plt.savefig(svdir +'/mcp_profiles_17_18_pH-NH4-DIC-Fe-DOC-SO4.pdf',bbox_inches ='tight')


In [ ]:
# process data to produce 2018 + 2019 MZA grids
# pH, SO4, TIC, Ca, DOC, Fe, Sr, Cl, Si + river concs + river elevations
fdir = '/Users/christiandewey/Library/CloudStorage/GoogleDrive-christian.w.dewey@gmail.com/My Drive/manuscripts/2023_Dewey-Fendorf-etal_meanders/data'

t = 'MZT1'
#positions = {'MCP1-1': 2, 'MCP1-2':17, 'MCP1-3': 32, 'MCP1-4': 48, 'MCP1-5': 63 }

positions = {'MZT1-1-D': "MZA1", 'MZT1-2-D':"MZA2", 'MZT1-3-D': "MZA3", 'MZT1-4-D': "MZA4", 'MZT1-5-D': "MZA5" }

## get 2019 MZA data
fname = '/porewater/mz_2019_porewater_ess-dive_replica.csv'
data1 = pd.read_csv(fdir + fname,parse_dates=['Date'])
data1['trans_pos_m'] = None

for r in range(len(data1)):
    w = data1['Well'].iloc[r]
    depth = data1['Depth name'].iloc[r]
    if (t in w) and (depth == 'Deep'):
        data1.loc[r,('trans_pos_m')]= positions[w]
    
sub19 = data1[data1['Well'].str.contains(t)==True].copy()
sub19['Ca_M'] = sub19['Ca_M']*1e3
sub19['Fe_M'] = sub19['Fe_M'].loc[:]*1e6
sub19['P_M'] = sub19['P_M']*1e6
sub19['SO4_mM'] = sub19['SO4_mM']*1e3
sub19['NO3_mM'] = sub19['NO3_mM']*1e3
sub19['NH4_mM'] = sub19['NH4_mM']*1e3
sub19['DOC_mM'] = sub19['DOC_mM']*1e3


## get 2018 MZA data
fname = '/porewater/mz_2018_porewater_ess-dive.csv'
data2 = pd.read_csv(fdir + fname,parse_dates=['Date'])
data2['trans_pos_m'] = None

for r in range(len(data2)):
    w = data2['Well'].iloc[r]
    depth = data2['Depth name'].iloc[r]
    if (t in w) and (depth == 'Deep'):
        data2.loc[r,('trans_pos_m')]= positions[w]
    
sub18 = data2[data2['Well'].str.contains(t)==True].copy()
sub18['Ca_M'] = sub18['Ca_M']*1e3
sub18['Fe_M'] = sub18['Fe_M'].loc[:]*1e6
sub18['P_M'] = sub18['P_M']*1e6
sub18['SO4_mM'] = sub18['SO4_mM']*1e3
sub18['NO3_mM'] = sub18['NO3_mM']*1e3
sub18['NH4_mM'] = sub18['NH4_mM']*1e3
sub18['DOC_mM'] = sub18['DOC_mM']*1e3



mza_18_19 = pd.concat([sub18,sub19],ignore_index=True)
mza_18_19['year'] = mza_18_19['Date'].dt.year.copy()
catpos = {'MZA1': 0, 'MZA2':1, 'MZA3': 2, 'MZA4': 3, 'MZA5': 4 }

def func(x):
    if x != None:
        v = catpos[x]
        return v

mza_18_19['CatPos'] = mza_18_19['trans_pos_m'].map(lambda x: func(x))

In [ ]:
# create MZA plots
plt.rcParams['ytick.major.pad']='1.5'
plt.rcParams['xtick.major.pad']='0.5'
xpltl = -.25
ypltl = 0.98

years = [2018,2019]

fig, ((ax1,ax3),(ax2,ax7),(ax5,ax8)) = plt.subplots(3,2, figsize = (6, 8))

makePanel('pH',mza_18_19,years, ax1)
ax1.set_ylabel('pH')
ax1.yaxis.set_major_locator(MultipleLocator(1))
ax1.set_ylim(6.6,8.3)

makePanel('Fe_M',mza_18_19,years, ax2)
ax2.set_ylabel('Fe ('+r'$\mu$M)')
#ax2.set_ylim(0,225)

makePanel('NH4_mM',mza_18_19,years, ax3)
ax3.set_ylabel('NH$_4^{+}$ ('+r'$\mu$M)')
ax3.yaxis.set_minor_locator(AutoMinorLocator())
ax3.set_ylim(bottom = 0)

makePanel('SO4_mM',mza_18_19,years, ax7)
ax7.set_ylabel('SO$_4^{2-}$ ('+r'$\mu$M)')
ax7.yaxis.set_minor_locator(AutoMinorLocator())
ax7.set_ylim(bottom = -5)

makePanel('DIC_mM',mza_18_19,years, ax5)
ax5.set_ylabel('DIC (mM)')
ax5.yaxis.set_major_locator(MultipleLocator(1))
ax5.yaxis.set_minor_locator(AutoMinorLocator())
ax5.set_ylim(bottom = 0)

makePanel('DOC_mM',mza_18_19,years, ax8)
ax8.set_ylabel('DOC ('+r'$\mu$M)')
ax8.set_ylim(bottom = 0)

'''makePanel('P_M',mcp17_18,years, ax4)
ax4.set_ylabel('TDP ('+r'$\mu$M)')
ax4.yaxis.set_minor_locator(AutoMinorLocator())
ax4.set_ylim(0,22)

makePanel('Ca_M',mcp17_18,years, ax6)
ax6.set_ylabel('Ca (mM)')
ax6.yaxis.set_major_locator(MultipleLocator(0.5))
ax6.set_ylim(0.9,2.6)
'''

ax = ax1 #pH 
ax.text(xpltl, ypltl,'a',
        horizontalalignment='center',
        verticalalignment='center',
        transform = ax.transAxes, fontweight='bold', fontsize = 12)

ax =ax3 # ammonium
ax.text(xpltl, ypltl,'b',
        horizontalalignment='center',
        verticalalignment='center',
        transform = ax.transAxes, fontweight='bold', fontsize = 12)

ax =ax5 # DIC
ax.text(xpltl, ypltl,'e',
        horizontalalignment='center',
        verticalalignment='center',
        transform = ax.transAxes, fontweight='bold', fontsize = 12)
'''ax =ax6 # calcium
ax.text(xpltl, ypltl,'d',
        horizontalalignment='center',
        verticalalignment='center',
        transform = ax.transAxes, fontweight='bold', fontsize = 12)'''

ax = ax2 #Fe
ax.text(xpltl, ypltl,'c',
        horizontalalignment='center',
        verticalalignment='center',
        transform = ax.transAxes, fontweight='bold', fontsize = 12)
ax = ax8 #DOC
ax.text(xpltl, ypltl,'f',
        horizontalalignment='center',
        verticalalignment='center',
        transform = ax.transAxes, fontweight='bold', fontsize = 12)

ax = ax7 #sulfate
ax.text(xpltl, ypltl,'d',
        horizontalalignment='center',
        verticalalignment='center',
        transform = ax.transAxes, fontweight='bold', fontsize = 12)
'''ax = ax4 #TDP
ax.text(xpltl, ypltl,'h',
        horizontalalignment='center',
        verticalalignment='center',
        transform = ax.transAxes, fontweight='bold', fontsize = 12)'''



makeColorBars([2018,2019],fig)

fig.tight_layout()

plt.savefig(svdir +'/mza_profiles_18_19_pH-NH4-DIC-Fe-DOC-SO4.pdf',bbox_inches ='tight')
